# Scanners

Adapted from [ib_async scanners](https://ib-api-reloaded.github.io/ib_async/notebooks.html).
Demonstrates scanner parameters and market scanner subscriptions.

In [ ]:
import os, threading, time
from types import SimpleNamespace
from dotenv import load_dotenv
from ibx import EWrapper, EClient, ContractDetails

load_dotenv()
USERNAME = os.environ["IB_USERNAME"]
PASSWORD = os.environ["IB_PASSWORD"]

In [ ]:
class App(EWrapper):
    def __init__(self):
        super().__init__()
        self.scanner_params_xml = None
        self.scanner_results = {}  # req_id -> list[(rank, con_id)]
        self.done = {}             # req_id -> Event
        self.params_done = threading.Event()
        self.connected = threading.Event()

    def next_valid_id(self, order_id):
        self.connected.set()

    def scanner_parameters(self, xml):
        self.scanner_params_xml = xml
        self.params_done.set()

    def scanner_data(self, req_id, rank, contract_details, distance, benchmark, projection, legs_str):
        self.scanner_results.setdefault(req_id, []).append(
            (rank, contract_details.contract.con_id)
        )

    def scanner_data_end(self, req_id):
        if req_id in self.done:
            self.done[req_id].set()

    def error(self, req_id, error_code, error_string, advanced_order_reject_json=""):
        if error_code not in (2104, 2106, 2158):
            print(f"Error {error_code}: {error_string}")

In [ ]:
app = App()
client = EClient(app)
client.connect(username=USERNAME, password=PASSWORD, paper=True)

thread = threading.Thread(target=client.run, daemon=True)
thread.start()
app.connected.wait(timeout=10)
print(f"Connected: {client.is_connected()}")

### Scanner Parameters

Request the XML document describing available scan types, instruments, and locations:

In [ ]:
client.req_scanner_parameters()
app.params_done.wait(timeout=10)

if app.scanner_params_xml:
    print(f"Scanner params XML length: {len(app.scanner_params_xml)} chars")
    # Show first 500 chars as preview
    print(app.scanner_params_xml[:500])
else:
    print("No scanner parameters received.")

### Run a Scanner

Subscribe to "Top % Gainers" on US major exchanges.
The subscription object needs `instrument`, `locationCode`, `scanCode`, and `numberOfRows` attributes:

In [ ]:
sub = SimpleNamespace(
    instrument="STK",
    locationCode="STK.US.MAJOR",
    scanCode="TOP_PERC_GAIN",
    numberOfRows=10,
)

app.done[1] = threading.Event()
app.scanner_results[1] = []

client.req_scanner_subscription(1, sub, [])
app.done[1].wait(timeout=15)

results = app.scanner_results.get(1, [])
print(f"Got {len(results)} scanner results")
for rank, con_id in results:
    print(f"  Rank {rank}: conId={con_id}")

### Cancel Scanner Subscription

In [ ]:
client.cancel_scanner_subscription(1)

### Resolve Contract Details for Scanner Results

The scanner returns conIds. Use `req_contract_details` to get symbols and names:

In [ ]:
from ibx import Contract

detail_results = {}  # req_id -> list[ContractDetails]
detail_done = {}     # req_id -> Event

# Patch in temporary callbacks for contract details
orig_cd = app.contract_details
orig_cd_end = app.contract_details_end

def on_cd(req_id, cd):
    detail_results.setdefault(req_id, []).append(cd)

def on_cd_end(req_id):
    if req_id in detail_done:
        detail_done[req_id].set()

app.contract_details = on_cd
app.contract_details_end = on_cd_end

for i, (rank, con_id) in enumerate(results[:5]):
    rid = 100 + i
    detail_done[rid] = threading.Event()
    detail_results[rid] = []
    c = Contract(con_id=con_id)
    client.req_contract_details(rid, c)

# Wait for all
for i in range(min(5, len(results))):
    detail_done[100 + i].wait(timeout=10)

print("Top gainers:")
for i, (rank, con_id) in enumerate(results[:5]):
    dets = detail_results.get(100 + i, [])
    if dets:
        cd = dets[0]
        print(f"  #{rank}: {cd.contract.symbol} — {cd.long_name} (conId={con_id})")
    else:
        print(f"  #{rank}: conId={con_id} (no details)")

# Restore original callbacks
app.contract_details = orig_cd
app.contract_details_end = orig_cd_end

In [ ]:
client.disconnect()